In [ ]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import clear_output
import time

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "vectordb",
    "user": "admin", "password": "password123"
}

def fetch_ohlc():
    conn = psycopg2.connect(**DB_CONFIG)
    df = pd.read_sql("""
        SELECT symbol, window_start, open, high, low, close, event_count
        FROM ohlc_results
        ORDER BY window_start DESC
        LIMIT 100
    """, conn)
    conn.close()
    return df

def fetch_anomalies():
    conn = psycopg2.connect(**DB_CONFIG)
    df = pd.read_sql("""
        SELECT symbol, window_start, price_change_pct, open, close
        FROM anomalies
        ORDER BY detected_at DESC
        LIMIT 20
    """, conn)
    conn.close()
    return df

# Live refresh loop — run this cell, it updates every 30 seconds
for _ in range(20):
    clear_output(wait=True)
    df = fetch_ohlc()
    anomalies = fetch_anomalies()

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    for symbol in df['symbol'].unique():
        sdf = df[df['symbol'] == symbol].sort_values('window_start')
        axes[0].plot(sdf['window_start'], sdf['close'], label=symbol, marker='o', markersize=3)

    axes[0].set_title('Close price per 1-min window')
    axes[0].set_xlabel('Window start')
    axes[0].set_ylabel('Price (USD)')
    axes[0].legend()
    axes[0].tick_params(axis='x', rotation=45)

    if not anomalies.empty:
        axes[1].bar(
            anomalies['symbol'] + '\n' + anomalies['window_start'].astype(str).str[:16],
            anomalies['price_change_pct'],
            color=['red' if x < 0 else 'green' for x in anomalies['price_change_pct']]
        )
        axes[1].axhline(y=2, color='orange', linestyle='--', label='+2% threshold')
        axes[1].axhline(y=-2, color='orange', linestyle='--', label='-2% threshold')
        axes[1].set_title('Anomalies detected (price move >= 2%)')
        axes[1].set_ylabel('Price change %')
        axes[1].legend()
        axes[1].tick_params(axis='x', rotation=45)
    else:
        axes[1].text(0.5, 0.5, 'No anomalies detected yet',
                    ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_title('Anomalies')

    plt.tight_layout()
    plt.show()
    print(f"Last refreshed: {pd.Timestamp.now().strftime('%H:%M:%S')} — refreshing every 30s")
    time.sleep(30)